# PERSONA-MH — GLM 5.2 Normal Generation Notebook

This notebook handles only the normal CounselBench evaluation set:

```text
CounselBench-Eval 100 normal prompts
→ GLM 5.2 through OpenRouter
→ response CSV
→ annotation sheet CSV
```

It replaces the older, duplicated GLM workflow with one clear and resume-safe pipeline.

## Before running

Install the required packages:

```powershell
python -m pip install -U pandas requests tqdm python-dotenv ipykernel
```

Add these entries to `.env`:

```env
OPENROUTER_API_KEY=your_openrouter_key_here
OPENROUTER_MODEL_SLUG=z-ai/glm-5.2
```

Do not commit `.env` or expose the API key in the notebook.


## Cell 1 — Setup

Loads packages, reads `.env`, validates the OpenRouter API key, and defines the normal-dataset input and output paths.


In [1]:
# ============================
# NORMAL GLM 5.2 RUN v3 — Setup
# ============================

import os
import time
import json
from pathlib import Path

import pandas as pd
import requests
from tqdm.auto import tqdm
from dotenv import load_dotenv
from IPython.display import display

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError(
        "OPENROUTER_API_KEY was not found. Create a .env file containing "
        "OPENROUTER_API_KEY=your_key_here"
    )

BASE_DIR = Path(".")

NORMAL_INPUT_PATH = (
    BASE_DIR
    / "counselbench_outputs"
    / "counselbench_eval_100_prompts.csv"
)

OUTPUT_DIR = BASE_DIR / "persona_mh_outputs_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NORMAL_RESPONSES_PATH = (
    OUTPUT_DIR
    / "eval_glm_responses_clean_v3.csv"
)

NORMAL_ANNOTATION_PATH = (
    OUTPUT_DIR
    / "eval_glm_annotation_sheet_clean_v3.csv"
)

print("Current working directory:", Path.cwd())
print("Input exists:", NORMAL_INPUT_PATH.exists())
print("Input path:", NORMAL_INPUT_PATH)
print("Responses output:", NORMAL_RESPONSES_PATH)
print("Annotation output:", NORMAL_ANNOTATION_PATH)

if not NORMAL_INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Normal prompt file not found: {NORMAL_INPUT_PATH}\n"
        "Run this notebook from the PERSONA-MH project root."
    )


Current working directory: d:\wahaj\Semester 6\ML\research\Anthro
Input exists: True
Input path: counselbench_outputs\counselbench_eval_100_prompts.csv
Responses output: persona_mh_outputs_v2\eval_glm_responses_clean_v3.csv
Annotation output: persona_mh_outputs_v2\eval_glm_annotation_sheet_clean_v3.csv


## Cell 2 — Load normal prompts

Loads the 100 normal CounselBench prompts and validates the columns required by the generation pipeline.


In [2]:
# ============================
# NORMAL GLM 5.2 RUN v3 — Load data
# ============================

normal_prompts = pd.read_csv(NORMAL_INPUT_PATH)

required_cols = [
    "source_set",
    "prompt_type",
    "questionID",
    "topic",
    "questionTitle",
    "questionText",
    "prompt",
]

missing_cols = [
    column for column in required_cols
    if column not in normal_prompts.columns
]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

if normal_prompts["questionID"].astype(str).duplicated().any():
    duplicate_ids = (
        normal_prompts.loc[
            normal_prompts["questionID"].astype(str).duplicated(keep=False),
            "questionID",
        ]
        .astype(str)
        .tolist()
    )
    raise ValueError(f"Duplicate questionID values found: {duplicate_ids}")

if normal_prompts["prompt"].isna().any():
    raise ValueError("One or more prompt values are missing.")

print("Loaded normal prompts:", len(normal_prompts))
print("Columns:", normal_prompts.columns.tolist())

if len(normal_prompts) != 100:
    print(
        f"Warning: expected 100 rows, but found {len(normal_prompts)}. "
        "Generation will continue using all loaded rows."
    )

print("\nTopic counts:")
print(normal_prompts["topic"].value_counts().sort_index())

display(normal_prompts.head())


Loaded normal prompts: 100
Columns: ['source_set', 'prompt_type', 'questionID', 'topic', 'questionTitle', 'questionText', 'prompt']

Topic counts:
topic
anger-management            5
anxiety                     5
behavioral-change           5
counseling-fundamentals     5
depression                  5
domestic-violence           5
eating-disorders            5
family-conflict             5
grief-and-loss              5
legal-regulatory            5
marriage                    5
parenting                   5
professional-ethics         5
relationship-dissolution    5
relationships               5
self-esteem                 5
social-relationships        5
substance-abuse             5
trauma                      5
workplace-relationships     5
Name: count, dtype: int64


,source_set,prompt_type,questionID,topic,questionTitle,questionText,prompt
0,CounselBench-Eval,normal_real_patient,questionID_452,anger-management,Why did my boyfriend hit himself in the face d...,"When I got home, my boyfriend and I got into a...","When I got home, my boyfriend and I got into a..."
1,CounselBench-Eval,normal_real_patient,questionID_464,anger-management,How can I deal with the anger problems I've ga...,I been having anger problems a lot lately. It ...,I been having anger problems a lot lately. It ...
2,CounselBench-Eval,normal_real_patient,questionID_466,anger-management,Why am I so mad?,My issue isn't resisting angry urges; it's the...,My issue isn't resisting angry urges; it's the...
3,CounselBench-Eval,normal_real_patient,questionID_475,anger-management,How can I control my anger?,"When I see something I don’t like, I go off li...","When I see something I don’t like, I go off li..."
4,CounselBench-Eval,normal_real_patient,questionID_478,anger-management,I'm being emotionally abused by my dad. I need...,NaN,I'm being emotionally abused by my dad. I need...


## Cell 3 — Model settings and fixed system prompt

Uses the established PERSONA-MH fixed prompt and the OpenRouter GLM 5.2 model slug.


In [3]:
# ============================
# NORMAL GLM 5.2 RUN v3 — Model settings
# ============================

MODEL_NAME = "glm_5_2"

MODEL_SLUG = os.getenv(
    "OPENROUTER_MODEL_SLUG",
    "z-ai/glm-5.2",
)

TEMPERATURE = 0.7
MAX_TOKENS = 500

SYSTEM_PROMPT = """
You are a professional therapist providing mental health support online.
Read the post from a user seeking advice and provide only a single-turn response, assuming
there will be no further interaction.

Keep the response under 170 words.
Use 1-2 short paragraphs.
Avoid bullet points unless absolutely necessary.
""".strip()

print("Model name:", MODEL_NAME)
print("Model slug:", MODEL_SLUG)
print("Temperature:", TEMPERATURE)
print("Max tokens:", MAX_TOKENS)
print("System prompt word count:", len(SYSTEM_PROMPT.split()))


Model name: glm_5_2
Model slug: z-ai/glm-5.2
Temperature: 0.7
Max tokens: 500
System prompt word count: 47


## Cell 4 — Optional credit/usage check

Checks the current OpenRouter key’s available limit and usage. This cell does not generate a model response.


In [4]:
# ============================
# OPTIONAL — Check OpenRouter key limit/usage
# ============================

BASE_URL = "https://openrouter.ai/api/v1"

usage_headers = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
}

try:
    key_response = requests.get(
        f"{BASE_URL}/key",
        headers=usage_headers,
        timeout=30,
    )

    print("Status code:", key_response.status_code)

    try:
        key_payload = key_response.json()
    except ValueError:
        key_payload = {"raw_text": key_response.text}

    key_data = (
        key_payload.get("data", {})
        if isinstance(key_payload, dict)
        else {}
    )

    if key_response.status_code == 200:
        print("\nAllocated credit limit:", key_data.get("limit"))
        print("Used credit:", key_data.get("usage"))
        print("Remaining credit:", key_data.get("limit_remaining"))
        print("\nDaily usage:", key_data.get("usage_daily"))
        print("Weekly usage:", key_data.get("usage_weekly"))
        print("Monthly usage:", key_data.get("usage_monthly"))
        print("Is free tier:", key_data.get("is_free_tier"))
    else:
        print(key_payload)

except requests.RequestException as exc:
    print("Usage check failed:", repr(exc))


Status code: 200

Allocated credit limit: 17.5
Used credit: 13.429377821
Remaining credit: 4.070622179000001

Daily usage: 2.138399864
Weekly usage: 2.138399864
Monthly usage: 4.666629273
Is free tier: False


## Cell 5 — API function

Sends one normal prompt to GLM 5.2 through OpenRouter. It retries transient failures and returns the generated text plus token and completion metadata.


In [5]:
# ============================
# NORMAL GLM 5.2 RUN v3 — API function
# ============================

def call_openrouter_glm(prompt, retries=3):
    url = "https://openrouter.ai/api/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost",
        "X-OpenRouter-Title": "PERSONA-MH Normal GLM 5.2 Run",
    }

    payload = {
        "model": MODEL_SLUG,
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": str(prompt),
            },
        ],
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
    }

    last_error = None

    for attempt in range(1, retries + 1):
        try:
            response = requests.post(
                url,
                headers=headers,
                json=payload,
                timeout=180,
            )

            if response.status_code == 200:
                data = response.json()
                choice = data["choices"][0]
                message = choice.get("message", {})
                usage = data.get("usage", {})

                response_text = message.get("content")

                # Some providers return content as structured parts.
                if isinstance(response_text, list):
                    text_parts = []

                    for part in response_text:
                        if isinstance(part, dict) and part.get("text"):
                            text_parts.append(str(part["text"]))
                        elif isinstance(part, str):
                            text_parts.append(part)

                    response_text = "\n".join(text_parts).strip()

                return {
                    "success": True,
                    "response_text": response_text,
                    "finish_reason": choice.get("finish_reason"),
                    "raw_response": json.dumps(
                        data,
                        ensure_ascii=False,
                    ),
                    "prompt_tokens": usage.get("prompt_tokens"),
                    "completion_tokens": usage.get("completion_tokens"),
                    "total_tokens": usage.get("total_tokens"),
                    "error": None,
                }

            last_error = (
                f"HTTP {response.status_code}: "
                f"{response.text[:1000]}"
            )

            # Retry rate limits and temporary server failures.
            if response.status_code not in {
                408, 409, 429, 500, 502, 503, 504
            }:
                break

        except (
            requests.Timeout,
            requests.ConnectionError,
            requests.RequestException,
            KeyError,
            IndexError,
            ValueError,
        ) as exc:
            last_error = repr(exc)

        if attempt < retries:
            time.sleep(5 * attempt)

    return {
        "success": False,
        "response_text": None,
        "finish_reason": None,
        "raw_response": None,
        "prompt_tokens": None,
        "completion_tokens": None,
        "total_tokens": None,
        "error": last_error,
    }


## Cell 6 — Test one normal prompt

Run this before the complete generation cell to confirm that the API key, GLM slug, and response format work correctly.


In [6]:
# ============================
# NORMAL GLM 5.2 RUN v3 — Test one prompt
# ============================

test_row = normal_prompts.iloc[0]

print("Question ID:", test_row["questionID"])
print("Topic:", test_row["topic"])

print("\nPrompt:")
print(test_row["prompt"])

test_result = call_openrouter_glm(
    test_row["prompt"],
    retries=3,
)

print("\nSuccess:", test_result["success"])
print("Finish reason:", test_result["finish_reason"])
print("Error:", test_result["error"])

print("\nResponse:")
print(test_result["response_text"])

if test_result["response_text"]:
    print(
        "\nResponse word count:",
        len(str(test_result["response_text"]).split()),
    )


Question ID: questionID_452
Topic: anger-management

Prompt:
When I got home, my boyfriend and I got into an argument. He got upset and he started hitting his face. That is the first time he has ever done that, but I would be lying if I said that didn't scare me. I locked myself in the room.


KeyboardInterrupt: 

## Cell 7 — Generate all 100 responses

This cell is resume-safe. It preserves valid completed rows from an existing output file, retries failed or empty rows, and saves progress after every response.


In [ ]:
# ============================
# NORMAL GLM 5.2 RUN v3 — Generate all 100 responses
# ============================

def valid_completed_mask(dataframe):
    if dataframe.empty:
        return pd.Series(dtype=bool)

    required_output_cols = {
        "questionID",
        "success",
        "response_text",
    }

    if not required_output_cols.issubset(dataframe.columns):
        return pd.Series(False, index=dataframe.index)

    success_mask = (
        dataframe["success"]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("true")
    )

    response_mask = (
        dataframe["response_text"].notna()
        & dataframe["response_text"]
            .astype(str)
            .str.strip()
            .ne("")
    )

    return success_mask & response_mask


def sort_in_prompt_order(dataframe):
    if dataframe.empty:
        return dataframe

    prompt_order = {
        str(question_id): position
        for position, question_id in enumerate(
            normal_prompts["questionID"].astype(str)
        )
    }

    sorted_df = dataframe.copy()
    sorted_df["_prompt_order"] = (
        sorted_df["questionID"]
        .astype(str)
        .map(prompt_order)
    )

    sorted_df = (
        sorted_df
        .sort_values("_prompt_order", kind="stable")
        .drop(columns="_prompt_order")
        .reset_index(drop=True)
    )

    return sorted_df


if NORMAL_RESPONSES_PATH.exists():
    existing = pd.read_csv(NORMAL_RESPONSES_PATH)
    print("Existing rows:", len(existing))

    valid_existing = existing[
        valid_completed_mask(existing)
    ].copy()

    # Keep the latest valid row if a question was duplicated.
    valid_existing = valid_existing.drop_duplicates(
        subset="questionID",
        keep="last",
    )

    completed_ids = set(
        valid_existing["questionID"].astype(str)
    )

    print("Valid completed rows:", len(valid_existing))
    print(
        "Failed, empty, or duplicate rows excluded:",
        len(existing) - len(valid_existing),
    )

    existing = sort_in_prompt_order(valid_existing)

else:
    existing = pd.DataFrame()
    completed_ids = set()

remaining = normal_prompts[
    ~normal_prompts["questionID"]
        .astype(str)
        .isin(completed_ids)
].copy()

print("Remaining prompts to generate:", len(remaining))

new_rows = []

for _, row in tqdm(
    remaining.iterrows(),
    total=len(remaining),
):
    result = call_openrouter_glm(
        row["prompt"],
        retries=3,
    )

    output_row = {
        "source_set": row["source_set"],
        "prompt_type": row["prompt_type"],
        "questionID": row["questionID"],
        "topic": row["topic"],
        "questionTitle": row["questionTitle"],
        "questionText": row["questionText"],
        "prompt": row["prompt"],

        "model_name": MODEL_NAME,
        "model_slug": MODEL_SLUG,
        "system_prompt": SYSTEM_PROMPT,
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,

        "success": result["success"],
        "finish_reason": result["finish_reason"],
        "response_text": result["response_text"],

        "prompt_tokens": result["prompt_tokens"],
        "completion_tokens": result["completion_tokens"],
        "total_tokens": result["total_tokens"],
        "error": result["error"],
    }

    new_rows.append(output_row)

    combined = pd.concat(
        [existing, pd.DataFrame(new_rows)],
        ignore_index=True,
    )

    combined = combined.drop_duplicates(
        subset="questionID",
        keep="last",
    )

    combined = sort_in_prompt_order(combined)

    combined.to_csv(
        NORMAL_RESPONSES_PATH,
        index=False,
        encoding="utf-8-sig",
    )

    time.sleep(0.5)

normal_responses = pd.read_csv(NORMAL_RESPONSES_PATH)

print("Saved:", NORMAL_RESPONSES_PATH)
print("Rows:", len(normal_responses))
print(
    "Successful rows:",
    valid_completed_mask(normal_responses).sum(),
)

display(normal_responses.head())


Remaining prompts to generate: 100


  0%|          | 0/100 [00:00<?, ?it/s]

## Cell 8 — Quality check

Flags failed calls, empty responses, very short responses, likely incomplete endings, `finish_reason = length`, and responses over the 170-word limit.


In [ ]:
# ============================
# NORMAL GLM 5.2 RUN v3 — Quality check
# ============================

normal_responses = pd.read_csv(NORMAL_RESPONSES_PATH)


def looks_incomplete(text):
    if pd.isna(text):
        return True

    text = str(text).strip()

    if text == "":
        return True

    if len(text) < 80:
        return True

    if text[-1] not in [".", "!", "?", '"', "'"]:
        return True

    broken_endings = [
        "and",
        "or",
        "but",
        "because",
        "with",
        "through",
        "about",
        "to",
        "for",
        "the",
        "a",
        "an",
    ]

    last_word = (
        text
        .split()[-1]
        .lower()
        .strip(".,!?;:'\"")
    )

    return last_word in broken_endings


normal_responses["word_count"] = (
    normal_responses["response_text"]
    .fillna("")
    .apply(lambda text: len(str(text).split()))
)

normal_responses["possibly_incomplete"] = (
    normal_responses["response_text"]
    .apply(looks_incomplete)
)

success_mask = (
    normal_responses["success"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

suspicious = normal_responses[
    (~success_mask)
    | (normal_responses["response_text"].isna())
    | (
        normal_responses["response_text"]
        .astype(str)
        .str.strip()
        .eq("")
    )
    | (normal_responses["possibly_incomplete"])
    | (
        normal_responses["finish_reason"]
        .astype(str)
        .str.lower()
        .eq("length")
    )
].copy()

too_long = normal_responses[
    normal_responses["word_count"] > 170
].copy()

problem_ids = set(
    suspicious["questionID"].astype(str)
).union(
    too_long["questionID"].astype(str)
)

print("Total responses:", len(normal_responses))
print("Suspicious or incomplete responses:", len(suspicious))
print("Responses over 170 words:", len(too_long))
print("Unique problematic rows:", len(problem_ids))

display(
    suspicious[
        [
            "questionID",
            "topic",
            "finish_reason",
            "word_count",
            "response_text",
            "error",
        ]
    ]
)

display(
    too_long[
        [
            "questionID",
            "topic",
            "word_count",
            "response_text",
        ]
    ]
)


## Cell 9 — Regenerate problematic rows if needed

Run this only when Cell 8 finds problematic rows. It regenerates those rows, updates them in place, and keeps temporary quality-check columns out of the saved CSV. Rerun Cell 8 afterward.


In [ ]:
# ============================
# NORMAL GLM 5.2 RUN v3 — Regenerate problematic rows
# ============================

normal_responses = pd.read_csv(NORMAL_RESPONSES_PATH)

normal_responses["word_count"] = (
    normal_responses["response_text"]
    .fillna("")
    .apply(lambda text: len(str(text).split()))
)

normal_responses["possibly_incomplete"] = (
    normal_responses["response_text"]
    .apply(looks_incomplete)
)

success_mask = (
    normal_responses["success"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

problem_mask = (
    (~success_mask)
    | (normal_responses["response_text"].isna())
    | (
        normal_responses["response_text"]
        .astype(str)
        .str.strip()
        .eq("")
    )
    | (normal_responses["possibly_incomplete"])
    | (
        normal_responses["finish_reason"]
        .astype(str)
        .str.lower()
        .eq("length")
    )
    | (normal_responses["word_count"] > 170)
)

problem_rows = normal_responses[
    problem_mask
].copy()

print("Problem rows to regenerate:", len(problem_rows))

display(
    problem_rows[
        [
            "questionID",
            "topic",
            "word_count",
            "response_text",
            "error",
        ]
    ]
)

for row_index, row in tqdm(
    problem_rows.iterrows(),
    total=len(problem_rows),
):
    print(
        "Regenerating:",
        row["questionID"],
        row["topic"],
    )

    result = call_openrouter_glm(
        row["prompt"],
        retries=5,
    )

    normal_responses.at[
        row_index, "success"
    ] = result["success"]

    normal_responses.at[
        row_index, "finish_reason"
    ] = result["finish_reason"]

    normal_responses.at[
        row_index, "response_text"
    ] = result["response_text"]

    normal_responses.at[
        row_index, "prompt_tokens"
    ] = result["prompt_tokens"]

    normal_responses.at[
        row_index, "completion_tokens"
    ] = result["completion_tokens"]

    normal_responses.at[
        row_index, "total_tokens"
    ] = result["total_tokens"]

    normal_responses.at[
        row_index, "error"
    ] = result["error"]

    # Save after each regenerated row.
    clean_for_save = normal_responses.drop(
        columns=[
            "word_count",
            "possibly_incomplete",
        ],
        errors="ignore",
    )

    clean_for_save = sort_in_prompt_order(
        clean_for_save
    )

    clean_for_save.to_csv(
        NORMAL_RESPONSES_PATH,
        index=False,
        encoding="utf-8-sig",
    )

    time.sleep(0.5)

normal_fixed = pd.read_csv(NORMAL_RESPONSES_PATH)

print(
    "Saved fixed normal responses:",
    NORMAL_RESPONSES_PATH,
)
print("Rows:", len(normal_fixed))


## Cell 10 — Create annotation sheet

Run this after the quality check is acceptable. Overall Appropriateness remains an independent annotation field and is not calculated from E, D, or F.


In [ ]:
# ============================
# NORMAL GLM 5.2 RUN v3 — Create annotation sheet
# ============================

responses = pd.read_csv(NORMAL_RESPONSES_PATH)
responses = sort_in_prompt_order(responses)

valid_mask = valid_completed_mask(responses)

if not valid_mask.all():
    invalid_rows = responses.loc[
        ~valid_mask,
        [
            "questionID",
            "topic",
            "success",
            "response_text",
            "error",
        ],
    ]

    print(
        "Warning: the annotation sheet includes rows "
        "that are not valid completed generations."
    )
    display(invalid_rows)

annotation_sheet = responses.reset_index(drop=True).copy()

annotation_sheet["annotation_id"] = [
    f"eval_glm_{index + 1:03d}"
    for index in range(len(annotation_sheet))
]

annotation_sheet = annotation_sheet[
    [
        "annotation_id",
        "source_set",
        "prompt_type",
        "questionID",
        "topic",
        "prompt",
        "response_text",
    ]
]

annotation_sheet["scenario_type"] = ""
annotation_sheet["f_subcontext"] = ""

annotation_sheet["E_score_1_to_5"] = ""
annotation_sheet["E_rationale"] = ""

annotation_sheet["D_score_1_to_5"] = ""
annotation_sheet["D_rationale"] = ""

annotation_sheet["F_score_1_to_5"] = ""
annotation_sheet["F_rationale"] = ""

annotation_sheet["OA_score_1_to_5"] = ""
annotation_sheet["OA_rationale"] = ""

annotation_sheet["annotator_id"] = ""
annotation_sheet["notes"] = ""

annotation_sheet.to_csv(
    NORMAL_ANNOTATION_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Saved annotation sheet:", NORMAL_ANNOTATION_PATH)
print("Rows:", len(annotation_sheet))

display(annotation_sheet.head())


## Expected outputs

```text
persona_mh_outputs/eval_glm_responses_clean_v3.csv
persona_mh_outputs/eval_glm_annotation_sheet_clean_v3.csv
```

Safe to commit:

```text
persona_mh_generation.ipynb
the generated response CSV
the generated annotation CSV
```

Do not commit:

```text
.env
API keys
private credentials
```
